# Análisis de Logs de Servidor
En la siguiente celda configuramos las dependencias, el modelo `ApacheLog` y la rutina de carga hacia un DataFrame.

In [1]:
#%pip install pandas

access_log = "tmpdir/logfile.txt"
access_log = "tmpdir/250k.log"


import pandas as pd
from pydantic import BaseModel, Field, IPvAnyAddress
from datetime import datetime
from typing import Optional
import apachelogs

# Definición del modelo Pydantic para la estructura del log
class ApacheLog(BaseModel):
    ip_address: IPvAnyAddress
    client_identity: Optional[str] = None
    userid: Optional[str] = None
    timestamp: datetime
    method: str
    request_uri: str
    http_version: str
    status_code: int = Field(ge=100, le=599)
    userAgent: Optional[str] = None
    referer: Optional[str] = None
    response_size: Optional[int] = None

def parse_apachelogs(line: str) -> Optional[dict]:
    try:
        log_format = '%h %l %u %t "%r" %s %b "%{Referer}i" "%{User-Agent}i" "%u"'
        parser = apachelogs.LogParser(log_format, errors=None)
        log_line_data = parser.parse(line)
        request_elements = log_line_data.request_line.split()
        method = request_elements[0] if len(request_elements) > 0 else "-"
        request_uri = request_elements[1] if len(request_elements) > 1 else "-"
        http_version = request_elements[2] if len(request_elements) > 2 else "-"

        # Validamos los datos y retornamos un diccionario para mayor eficiencia en Pandas
        return ApacheLog(
            ip_address=log_line_data.remote_host,
            client_identity=log_line_data.remote_logname,
            userid=log_line_data.remote_user,
            timestamp=log_line_data.request_time,
            method=method,
            request_uri=request_uri,
            http_version=http_version,
            status_code=log_line_data.status,
            # Usar .get() previene KeyError si el header no existe
            userAgent=log_line_data.headers_in.get("User-Agent"),
            referer=log_line_data.headers_in.get("Referer"),
            response_size=log_line_data.bytes_sent
        )

    except Exception:
        return None



In [2]:
from tqdm.auto import tqdm
tqdm.pandas(desc="Notebook Progress")
def cargar_datos(ruta: str) -> pd.DataFrame:
    datos = []
    with open(ruta, 'r', encoding='utf-8') as f:
        for linea in f:
            if linea.strip():
                parseado = parse_apachelogs(linea)
                if parseado:
                    datos.append(parseado)
    headers = list(ApacheLog.model_fields.keys())
    print(headers)
    df = pd.DataFrame([item.model_dump() for item in datos],  columns=headers)
    #df = pd.DataFrame(datos)
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
        #df.set_index('timestamp', inplace=True)
    return df


df = cargar_datos(access_log)
df.head()

['ip_address', 'client_identity', 'userid', 'timestamp', 'method', 'request_uri', 'http_version', 'status_code', 'userAgent', 'referer', 'response_size']


,ip_address,client_identity,userid,timestamp,method,request_uri,http_version,status_code,userAgent,referer,response_size
0,109.169.248.247,None,None,2015-12-12 17:25:11+00:00,GET,/administrator/,HTTP/1.1,200,Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20...,NaN,4263.0
1,109.169.248.247,None,None,2015-12-12 17:25:11+00:00,POST,/administrator/index.php,HTTP/1.1,200,Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20...,http://almhuette-raith.at/administrator/,4494.0
2,46.72.177.4,None,None,2015-12-12 17:31:08+00:00,GET,/administrator/,HTTP/1.1,200,Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20...,NaN,4263.0
3,46.72.177.4,None,None,2015-12-12 17:31:08+00:00,POST,/administrator/index.php,HTTP/1.1,200,Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20...,http://almhuette-raith.at/administrator/,4494.0
4,83.167.113.100,None,None,2015-12-12 17:31:25+00:00,GET,/administrator/,HTTP/1.1,200,Mozilla/5.0 (Windows NT 6.0; rv:34.0) Gecko/20...,NaN,4263.0


*Bis
Describamos

In [3]:
df.describe()


,status_code,response_size
count,249999.000000,2.486590e+05
mean,209.286701,7.990517e+04
std,41.940159,1.318444e+06
min,200.000000,0.000000e+00
25%,200.000000,4.263000e+03
50%,200.000000,4.481000e+03
75%,200.000000,4.498000e+03
max,501.000000,5.255139e+07


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 249999 entries, 0 to 249998
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   ip_address       249999 non-null  object             
 1   client_identity  0 non-null       object             
 2   userid           0 non-null       object             
 3   timestamp        249999 non-null  datetime64[us, UTC]
 4   method           249999 non-null  str                
 5   request_uri      249999 non-null  str                
 6   http_version     249999 non-null  str                
 7   status_code      249999 non-null  int64              
 8   userAgent        215252 non-null  str                
 9   referer          111488 non-null  str                
 10  response_size    248659 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(1), int64(1), object(3), str(5)
memory usage: 21.0+ MB


In [5]:
df.describe

<bound method NDFrame.describe of              ip_address client_identity userid                 timestamp  \
0       109.169.248.247            None   None 2015-12-12 17:25:11+00:00   
1       109.169.248.247            None   None 2015-12-12 17:25:11+00:00   
2           46.72.177.4            None   None 2015-12-12 17:31:08+00:00   
3           46.72.177.4            None   None 2015-12-12 17:31:08+00:00   
4        83.167.113.100            None   None 2015-12-12 17:31:25+00:00   
...                 ...             ...    ...                       ...   
249994  177.134.189.192            None   None 2016-06-16 12:48:43+00:00   
249995   200.148.166.42            None   None 2016-06-16 12:48:43+00:00   
249996   200.148.166.42            None   None 2016-06-16 12:48:44+00:00   
249997  177.134.189.192            None   None 2016-06-16 12:48:44+00:00   
249998  177.134.189.192            None   None 2016-06-16 12:48:44+00:00   

       method               request_uri http_version 

## 1. Direcciones IP más frecuentes
A continuación, identificaremos las direcciones IP que realizan más peticiones. Espero que esta información facilite su inspección de tráfico.

In [6]:
# Contamos las ocurrencias de cada IP y mostramos el top 10
top_ips = df['ip_address'].value_counts().head(10)
top_ips

ip_address
205.167.170.15     31844
134.249.53.185     17904
192.227.172.158    13474
79.142.95.122       3207
37.1.206.196        2474
148.251.50.49       1929
200.148.166.42      1526
91.200.12.22        1222
78.186.191.187       753
52.22.118.215        734
Name: count, dtype: int64

## 2. URLs más solicitadas
Procedemos ahora a extraer los recursos más demandados por los clientes en este conjunto de datos. 

In [7]:
# Frecuencia de los identificadores de recursos solicitados
top_urls = df['request_uri'].value_counts().head(10)
top_urls

request_uri
/administrator/index.php                                81972
/administrator/                                         54004
http://almhuette-raith.at/administrator/index.php       17912
/                                                        5834
/templates/_system/css/general.css                       3960
/apache-log/access.log                                   2374
/robots.txt                                              1934
/favicon.ico                                             1103
/media/system/js/mootools.js                             1077
/modules/mod_bowslideshow/tmpl/js/sliderman.1.3.0.js     1067
Name: count, dtype: int64

## 3. Top combinaciones (IP y URL)
Para un análisis más preciso, cruzaremos las IPs con las URLs solicitadas. Esto nos mostrará el interés específico de cada origen.

In [8]:
# Agrupamos por URI e IP para obtener los pares más comunes
top_combinaciones = df.value_counts(subset=['ip_address', 'request_uri']).head(10)
top_combinaciones

ip_address       request_uri                                      
134.249.53.185   http://almhuette-raith.at/administrator/index.php    17904
192.227.172.158  /administrator/index.php                             13474
79.142.95.122    /administrator/index.php                              3206
37.1.206.196     /administrator/index.php                              2469
148.251.50.49    /administrator/index.php                              1929
200.148.166.42   /administrator/index.php                              1526
91.200.12.22     /administrator/index.php                              1217
78.186.191.187   /administrator/index.php                               753
205.167.170.15   /                                                      556
192.111.153.243  /administrator/index.php                               508
Name: count, dtype: int64

## 4. Direcciones IP con más códigos de error
Filtraremos las respuestas que resultaron en error (códigos 400 en adelante) para aislar las IPs problemáticas. Una disculpa si estos números son altos.

In [9]:
# Filtramos errores y agrupamos por IP
errores_df = df[df['status_code'] >= 400]
ips_con_errores = errores_df['ip_address'].value_counts().head(10)
ips_con_errores

ip_address
205.167.170.15     899
52.22.118.215      732
94.219.251.26      233
84.58.165.21       147
188.127.233.220    119
84.112.161.41       98
76.168.128.240      91
73.231.163.153      60
46.161.9.8          50
95.91.205.238       44
Name: count, dtype: int64

## 5. URLs con más códigos de error
De la misma manera, buscaremos qué rutas en el servidor generan el mayor número de incidencias o respuestas fallidas.

In [10]:
# Contamos los URIs dentro del subconjunto de errores
urls_con_errores = errores_df['request_uri'].value_counts().head(10)
urls_con_errores

request_uri
/templates/_system/css/general.css                            3960
/favicon.ico                                                  1103
/wp-login.php                                                 1048
/wp-login.php?action=register                                  257
/index.php?option=com_easyblog&view=dashboard&layout=write     208
/libraries/joomla/exporter.php                                 114
/templates/jp_hotel/images/black_trans.png                      62
/templates/jp_hotel/images/black_suckerfish.png                 61
/templates/jp_hotel/images/hdot.gif                             61
/templates/jp_hotel/css/images/bg_raith.jpg                     49
Name: count, dtype: int64

## 6. URLs con más errores, desglosadas por IP
Calcularemos cuáles son los fallos más comunes cometidos por cada IP específica. Le presento los resultados agrupados.

In [11]:
# Agrupamos los errores por IP y luego por URL, obteniendo los principales fallos por individuo
errores_por_ip = errores_df.groupby('ip_address')['request_uri'].value_counts()
# Mostramos solo un resumen del top 10 general por brevedad
errores_por_ip.sort_values(ascending=False).head(10)

ip_address      request_uri                                    
205.167.170.15  /templates/_system/css/general.css                 174
84.58.165.21    /templates/_system/css/general.css                 145
205.167.170.15  /templates/jp_hotel/images/black_trans.png          58
                /templates/jp_hotel/images/black_suckerfish.png     57
                /templates/jp_hotel/images/hdot.gif                 57
84.112.161.41   /templates/_system/css/general.css                  49
                /favicon.ico                                        49
205.167.170.15  /templates/jp_hotel/css/images/bg_raith.jpg         48
73.231.163.153  /templates/_system/css/general.css                  43
95.91.205.238   /templates/_system/css/general.css                  40
Name: count, dtype: int64

## 7. Análisis de tráfico por ventana temporal (ej. 10 segundos)
Procederemos a examinar el volumen de peticiones agrupadadas en una ventana de 10 segundos por IP, útil para detectar posibles comportamientos abusivos.

In [12]:
# Establecemos el timestamp como índice y remuestreamos a ventanas de 10 segundos ('10S')
#df_temp = df.set_index('timestamp')
sdf = df.iloc[0:50000].copy()
df_temp = sdf.set_index(pd.DatetimeIndex(sdf["timestamp"]))
# Agrupamos por IP y ventana de tiempo, contando las peticiones
peticiones_por_ventana = df_temp.groupby('ip_address').resample('10s').size()
# Mostramos los periodos más intensos
top_ventanas = peticiones_por_ventana.nlargest(10)
top_ventanas

ip_address       timestamp                
205.167.170.15   2016-01-25 20:26:30+00:00    252
                 2016-01-28 20:27:50+00:00    208
                 2016-01-28 20:28:00+00:00    165
                 2016-02-11 19:16:10+00:00    162
                 2016-02-10 16:43:40+00:00    155
                 2016-02-10 17:23:40+00:00    141
                 2016-02-10 17:25:00+00:00    140
                 2016-02-10 17:06:30+00:00    131
                 2016-02-10 17:00:00+00:00    125
188.127.233.220  2016-01-22 08:35:50+00:00     86
dtype: int64

In [18]:
time_series = df_temp.resample('1ME').size()

print(time_series)

timestamp
2015-12-31 00:00:00+00:00    14149
2016-01-31 00:00:00+00:00    28227
2016-02-29 00:00:00+00:00     7624
Freq: ME, dtype: int64


## 8. Identificación de solicitudes anormales
Finalmente, intentaremos detectar anomalías. Para este ejercicio, definimos una anomalía como un error de servidor (5xx) o transferencias de datos atípicamente grandes.

In [ ]:
# Calculamos el percentil 99 del tamaño de respuesta para definir peticiones inusualmente pesadas
umbral_tamano = df['response_size'].quantile(0.99) if not df['response_size'].isnull().all() else 0

# Consideramos anormal: Errores internos de servidor o tamaño excesivo
anormales = df[(df['status_code'] >= 500) | (df['response_size'] > umbral_tamano)]

anormales[['ip_address', 'status_code', 'response_size', 'request_uri']].head(10)

,ip_address,status_code,response_size,request_uri
404,83.167.113.100,500,88.0,/administrator/
405,109.197.194.109,500,88.0,/administrator/
1643,103.27.238.252,200,333094.0,/apache-log/access.log
1729,158.222.15.22,200,349927.0,/apache-log/access.log
1739,180.180.100.148,200,352024.0,/apache-log/access.log
1740,175.136.239.174,200,352289.0,/apache-log/access.log
1754,178.19.111.195,200,355213.0,/apache-log/access.log
1769,176.49.68.156,500,88.0,/administrator/
1770,78.171.222.103,200,358210.0,/apache-log/access.log
1780,192.99.71.135,200,360390.0,/apache-log/access.log
